In [16]:
import numpy as np
from src.config import DATA_PROCESSED_DIR, G_2_PATH, T_LONG
from src.predictors import extract_influence_vector
from src.data_loader import load_temporal_edgelist

In [17]:
saved = np.load(DATA_PROCESSED_DIR / "trajectories_g2.npz")
nodes, trajectories = saved["nodes"], saved["trajectories"]

influence = extract_influence_vector(trajectories, T_LONG)

influence_counts = {}
for i in influence:
    if i in influence_counts:
        influence_counts[i] += 1
    else:
        influence_counts[i] = 1

for (influence, count) in sorted(influence_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"Influence: {influence}, Count: {count}")

Influence: 402, Count: 401
Influence: 401, Count: 1
Influence: 1, Count: 1


Since $N=403$, the influence of $402$ for most of the nodes and influence of $1$ for one node suggests the possibility that in $G_2$, one node is disconnected from the graph for the temporal network up to initial $1200$ time steps. Since the graph is connected, it is guaranteed that the first contact exists, but may fall outside of the specified horizon.

In [18]:
g = load_temporal_edgelist(G_2_PATH)

first_contacts = {}
for i, row in g.iterrows():
    u = row["u"]
    v = row["v"]
    t = row["t"]
    if (u in first_contacts and t < first_contacts[u]) or u not in first_contacts:
        first_contacts[u] = t
    if (v in first_contacts and t < first_contacts[v]) or v not in first_contacts:
        first_contacts[v] = t

for (node, timestep) in sorted(first_contacts.items(), key=lambda x: x[1], reverse=True):
    if timestep > T_LONG:
        print(f"Node ID: {node}, Time Step: {timestep}")
    else:
        break

Node ID: 391, Time Step: 2097


Node with ID $391$ makes its first contact in the $G_2$ network at $t=2097$, which is after $T\_LONG$, hence it's unreachable.